<a href="https://colab.research.google.com/github/selva-mani-007/gen-ai/blob/main/text_to_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from diffusers import DiffusionPipeline
import torch
import numpy as np
import imageio
from PIL import Image

# 📝 Ask user for text prompt
prompt = input("Enter a prompt for the video (e.g., 'astronaut dancing on Mars'): ")

# ⚙️ Load the model
pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# 🎬 Generate video frames (few steps to keep it short)
video_batches = pipe(prompt, num_inference_steps=25).frames  # 2–3 second clip

# 🎯 Resize config
target_resolution = (512, 512)
processed_frames = []

# 🖼️ Frame normalization & resizing
for batch in video_batches:
    for frame in batch:
        if frame.dtype != np.uint8:
            frame = (frame * 255).clip(0, 255).astype(np.uint8)
        if frame.ndim == 2:
            frame = np.stack([frame] * 3, axis=-1)
        elif frame.ndim == 3 and frame.shape[2] == 1:
            frame = np.repeat(frame, 3, axis=2)
        elif frame.ndim == 3 and frame.shape[2] > 4:
            frame = frame[:, :, :3]
        image = Image.fromarray(frame).resize(target_resolution, Image.LANCZOS)
        processed_frames.append(np.array(image))

print(f"✅ Total processed frames: {len(processed_frames)}")

# 💾 Save video using FFmpeg
output_path = "generated_video.mp4"
writer = imageio.get_writer(
    output_path,
    fps=8,
    codec='libx264',
    bitrate="5M",
    quality=10
)
for frame in processed_frames:
    writer.append_data(frame)
writer.close()

print(f"🎥 Video saved as {output_path}")


Enter a prompt for the video (e.g., 'astronaut dancing on Mars'): leo flying like monkey


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

✅ Total processed frames: 16
🎥 Video saved as generated_video.mp4
